In [9]:
import pandas as pd

holiday_data = pd.read_csv('raw data/holiday_list.csv')
outlet_coordinates = pd.read_csv('raw data/outlet_coordinates.csv')
outlet_master = pd.read_csv('raw data/outlet_master.csv')
transaction_data = pd.read_csv('raw data/transactions_history_final.csv')
distributor = pd.read_csv('raw data/distributor_seasonality_details.csv')


In [10]:
from pathlib import Path
import json
import pandas as pd

# Summaries output folder
outdir = Path('summaries')
outdir.mkdir(parents=True, exist_ok=True)

p = Path('raw data')
csvs = sorted(p.glob('*.csv'))

print('Found CSV files:', [str(x) for x in csvs])

dataset_rows = []
column_rows = []

for f in csvs:
    size_bytes = f.stat().st_size
    size_mb = size_bytes / 1024 / 1024
    fname = str(f)
    try:
        df = pd.read_csv(f, low_memory=False)
    except Exception:
        df = pd.read_csv(f, engine='python', low_memory=False)
    rows, cols = df.shape
    dataset_rows.append({'file': fname, 'rows': int(rows), 'cols': int(cols), 'size_mb': round(size_mb, 2)})
    for col in df.columns:
        s = df[col]
        missing = int(s.isna().sum())
        unique = int(s.nunique(dropna=True))
        unique_ratio = round(unique / rows, 4) if rows > 0 else None
        missing_ratio = round(missing / rows, 4) if rows > 0 else None
        dtype = str(s.dtype)
        try:
            sample_vals = s.dropna().unique()[:5].tolist()
        except Exception:
            sample_vals = []
        column_rows.append({
            'file': fname,
            'column': col,
            'dtype': dtype,
            'unique_count': unique,
            'unique_ratio': unique_ratio,
            'missing_count': missing,
            'missing_ratio': missing_ratio,
            'sample_values': json.dumps(sample_vals)
        })

# Write summaries into summaries/ folder
pd.DataFrame(dataset_rows).to_csv(outdir / 'dataset_summary.csv', index=False)
pd.DataFrame(column_rows).to_csv(outdir / 'column_summary.csv', index=False)

print('Wrote', outdir / 'dataset_summary.csv', 'and', outdir / 'column_summary.csv')


Found CSV files: ['raw data\\distributor_seasonality_details.csv', 'raw data\\holiday_list.csv', 'raw data\\outlet_coordinates.csv', 'raw data\\outlet_master.csv', 'raw data\\transactions_history_final.csv']
Wrote summaries\dataset_summary.csv and summaries\column_summary.csv


In [11]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

RAW_DIR = Path("raw data")
OUT_DIR = Path("summaries")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOW_CONFIDENCE_THRESHOLD = 0.60
RANDOM_STATE = 42


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def find_file(possible_names, base_dir):
    for name in possible_names:
        path = base_dir / name
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find any of these files: {possible_names}")


def find_common_key(df1, df2):
    possible_keys = ["Outlet_ID", "OutletId", "outlet_id", "OutletID", "outletid"]
    for key in possible_keys:
        if key in df1.columns and key in df2.columns:
            return key
    return None


def clean_column_names(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    return df


def standardize_outlet_size(series):
    cleaned = (
        series.astype("string")
        .str.strip()
        .str.title()
    )

    cleaned = cleaned.replace({
        "": np.nan,
        "Nan": np.nan,
        "None": np.nan,
        "Null": np.nan,
        "N/A": np.nan,
        "Na": np.nan
    })

    return cleaned


def safe_mode(x):
    mode_val = x.mode()
    if len(mode_val) > 0:
        return mode_val.iloc[0]
    return np.nan


# ============================================================
# LOAD DATA
# ============================================================

outlet_master_path = find_file(
    ["outlet_master.csv", "outlet_master_final.csv"],
    RAW_DIR
)

outlet_coordinates_path = find_file(
    ["outlet_coordinates.csv", "outlet_coordinates_final.csv"],
    RAW_DIR
)

transactions_path = find_file(
    ["transactions_history.csv", "transactions_history_final.csv"],
    RAW_DIR
)

om = clean_column_names(pd.read_csv(outlet_master_path))
oc = clean_column_names(pd.read_csv(outlet_coordinates_path))
tx = clean_column_names(pd.read_csv(transactions_path))

print("Loaded files:")
print("Outlet master:", outlet_master_path)
print("Outlet coordinates:", outlet_coordinates_path)
print("Transactions:", transactions_path)


# ============================================================
# MERGE OUTLET MASTER + COORDINATES
# ============================================================

key = find_common_key(om, oc)

if key is None:
    raise ValueError("No common Outlet_ID key found between outlet_master and outlet_coordinates.")

merged = om.merge(oc, on=key, how="left", suffixes=("", "_coords"))

target_col = next((c for c in merged.columns if "size" in c.lower()), None)

if target_col is None:
    raise ValueError('No column containing "size" found in outlet_master.')

merged[target_col] = standardize_outlet_size(merged[target_col])

print("\nTarget column:", target_col)
print("Join key:", key)


# ============================================================
# BASIC NUMERIC CLEANING
# ============================================================

if "Cooler_Count" in merged.columns:
    merged["Cooler_Count"] = pd.to_numeric(merged["Cooler_Count"], errors="coerce")

lat_col = next((c for c in merged.columns if "lat" in c.lower()), None)
lon_col = next((c for c in merged.columns if "lon" in c.lower() or "long" in c.lower()), None)

if lat_col is None or lon_col is None:
    raise ValueError("Latitude / Longitude columns not found.")

merged[lat_col] = pd.to_numeric(merged[lat_col], errors="coerce")
merged[lon_col] = pd.to_numeric(merged[lon_col], errors="coerce")


# ============================================================
# TRANSACTION-BASED FEATURE ENGINEERING
# ============================================================

required_tx_cols = [key, "Year", "Month", "Volume_Liters", "Total_Bill_Value"]

missing_cols = [c for c in required_tx_cols if c not in tx.columns]
if missing_cols:
    raise ValueError(f"Transactions file is missing required columns: {missing_cols}")

tx["Year"] = pd.to_numeric(tx["Year"], errors="coerce")
tx["Month"] = pd.to_numeric(tx["Month"], errors="coerce")
tx["Volume_Liters"] = pd.to_numeric(tx["Volume_Liters"], errors="coerce")
tx["Total_Bill_Value"] = pd.to_numeric(tx["Total_Bill_Value"], errors="coerce")

# Remove impossible transaction values only for feature engineering
tx_clean = tx[
    (tx["Year"].notna()) &
    (tx["Month"].between(1, 12)) &
    (tx["Volume_Liters"].notna()) &
    (tx["Volume_Liters"] >= 0) &
    (tx["Total_Bill_Value"].notna()) &
    (tx["Total_Bill_Value"] >= 0)
].copy()

tx_clean["Month_Index"] = tx_clean["Year"] * 12 + tx_clean["Month"]

# Monthly outlet level sales
monthly_agg_dict = {
    "monthly_liters": ("Volume_Liters", "sum"),
    "monthly_bill_value": ("Total_Bill_Value", "sum")
}

if "SKU_ID" in tx_clean.columns:
    monthly_agg_dict["monthly_sku_count"] = ("SKU_ID", "nunique")

monthly = (
    tx_clean.groupby([key, "Year", "Month", "Month_Index"])
    .agg(**monthly_agg_dict)
    .reset_index()
)

if "monthly_sku_count" not in monthly.columns:
    monthly["monthly_sku_count"] = np.nan

latest_month_index = monthly["Month_Index"].max()

monthly["is_last_3_months"] = monthly["Month_Index"] >= latest_month_index - 2
monthly["is_last_6_months"] = monthly["Month_Index"] >= latest_month_index - 5

# Main outlet-level sales features
tx_features = (
    monthly.groupby(key)
    .agg(
        avg_monthly_liters=("monthly_liters", "mean"),
        median_monthly_liters=("monthly_liters", "median"),
        max_monthly_liters=("monthly_liters", "max"),
        p75_monthly_liters=("monthly_liters", lambda x: np.percentile(x, 75)),
        p90_monthly_liters=("monthly_liters", lambda x: np.percentile(x, 90)),
        sales_std=("monthly_liters", "std"),
        total_liters=("monthly_liters", "sum"),
        avg_bill_value=("monthly_bill_value", "mean"),
        max_bill_value=("monthly_bill_value", "max"),
        total_bill_value=("monthly_bill_value", "sum"),
        avg_sku_count=("monthly_sku_count", "mean"),
        max_sku_count=("monthly_sku_count", "max"),
        active_months=("monthly_liters", "count")
    )
    .reset_index()
)

# Sales consistency: lower CV means more stable sales
tx_features["sales_cv"] = tx_features["sales_std"] / tx_features["avg_monthly_liters"].replace(0, np.nan)

# Recent 3-month average
recent_3 = (
    monthly[monthly["is_last_3_months"]]
    .groupby(key)
    .agg(recent_3_month_avg_liters=("monthly_liters", "mean"))
    .reset_index()
)

# Recent 6-month average
recent_6 = (
    monthly[monthly["is_last_6_months"]]
    .groupby(key)
    .agg(recent_6_month_avg_liters=("monthly_liters", "mean"))
    .reset_index()
)

# January-specific behavior
jan_features = (
    monthly[monthly["Month"] == 1]
    .groupby(key)
    .agg(
        avg_january_liters=("monthly_liters", "mean"),
        max_january_liters=("monthly_liters", "max")
    )
    .reset_index()
)

tx_features = tx_features.merge(recent_3, on=key, how="left")
tx_features = tx_features.merge(recent_6, on=key, how="left")
tx_features = tx_features.merge(jan_features, on=key, how="left")

# SKU variety across full history
if "SKU_ID" in tx_clean.columns:
    sku_features = (
        tx_clean.groupby(key)
        .agg(total_unique_skus=("SKU_ID", "nunique"))
        .reset_index()
    )
    tx_features = tx_features.merge(sku_features, on=key, how="left")

# Main distributor per outlet
if "Distributor_ID" in tx_clean.columns:
    dist_features = (
        tx_clean.groupby(key)
        .agg(
            Distributor_ID=("Distributor_ID", safe_mode),
            distributor_count=("Distributor_ID", "nunique")
        )
        .reset_index()
    )
    tx_features = tx_features.merge(dist_features, on=key, how="left")

# Merge transaction features into outlet master
merged = merged.merge(tx_features, on=key, how="left")

print("\nTransaction features added.")


# ============================================================
# FEATURE SELECTION
# ============================================================

numeric_candidates = [
    lat_col,
    lon_col,
    "Cooler_Count",

    "avg_monthly_liters",
    "median_monthly_liters",
    "max_monthly_liters",
    "p75_monthly_liters",
    "p90_monthly_liters",
    "sales_std",
    "sales_cv",
    "total_liters",

    "avg_bill_value",
    "max_bill_value",
    "total_bill_value",

    "avg_sku_count",
    "max_sku_count",
    "total_unique_skus",
    "active_months",

    "recent_3_month_avg_liters",
    "recent_6_month_avg_liters",

    "avg_january_liters",
    "max_january_liters",

    "distributor_count"
]

categorical_candidates = [
    "Outlet_Type",
    "Distributor_ID"
]

numeric_features = [c for c in numeric_candidates if c in merged.columns]
categorical_features = [c for c in categorical_candidates if c in merged.columns]

print("\nNumeric features used:")
print(numeric_features)

print("\nCategorical features used:")
print(categorical_features)


# ============================================================
# TRAIN / MISSING SPLIT
# ============================================================

train = merged[merged[target_col].notna()].copy()
missing = merged[merged[target_col].isna()].copy()

print("\nTraining rows:", len(train))
print("Missing rows to predict:", len(missing))

print("\nOutlet_Size class distribution:")
print(train[target_col].value_counts())

print("\nOutlet_Size class distribution %:")
print((train[target_col].value_counts(normalize=True) * 100).round(2))


# ============================================================
# PREPARE MODEL DATA
# ============================================================

X = train[numeric_features + categorical_features]
y = train[target_col].astype(str)

le = LabelEncoder()
y_enc = le.fit_transform(y)

print("\nEncoded classes:")
for cls, enc in zip(le.classes_, le.transform(le.classes_)):
    print(f"{cls}: {enc}")


# ============================================================
# PREPROCESSING + MODEL
# ============================================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

xgb = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    n_estimators=400,
    max_depth=4,
    learning_rate=0.04,
    subsample=0.90,
    colsample_bytree=0.90,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", xgb)
    ]
)


# ============================================================
# MANUAL STRATIFIED CROSS-VALIDATION WITH CLASS BALANCING
# ============================================================

min_class_count = train[target_col].value_counts().min()
n_splits = min(5, min_class_count)

oof_pred = np.zeros(len(X), dtype=int)
fold_results = []

if n_splits >= 2:
    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y_enc), start=1):
        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_tr = y_enc[train_idx]
        y_val = y_enc[val_idx]

        fold_model = clone(model)

        sample_weights = compute_sample_weight(
            class_weight="balanced",
            y=y_tr
        )

        fold_model.fit(
            X_tr,
            y_tr,
            classifier__sample_weight=sample_weights
        )

        pred_val = fold_model.predict(X_val)
        oof_pred[val_idx] = pred_val

        acc = accuracy_score(y_val, pred_val)
        macro_f1 = f1_score(y_val, pred_val, average="macro")

        fold_results.append({
            "fold": fold,
            "accuracy": acc,
            "macro_f1": macro_f1
        })

        print(f"Fold {fold}: Accuracy={acc*100:.2f}% | Macro F1={macro_f1*100:.2f}%")

    fold_results_df = pd.DataFrame(fold_results)

    print("\nCross-validation summary:")
    print(f"Accuracy: {fold_results_df['accuracy'].mean()*100:.2f}% ± {fold_results_df['accuracy'].std()*100:.2f}%")
    print(f"Macro F1: {fold_results_df['macro_f1'].mean()*100:.2f}% ± {fold_results_df['macro_f1'].std()*100:.2f}%")

    print("\nClassification report:")
    report_text = classification_report(
        y_enc,
        oof_pred,
        target_names=le.classes_
    )
    print(report_text)

    cm = confusion_matrix(y_enc, oof_pred)
    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual_{c}" for c in le.classes_],
        columns=[f"Pred_{c}" for c in le.classes_]
    )

    fold_results_df.to_csv(OUT_DIR / "outlet_size_imputation_cv_results.csv", index=False)
    cm_df.to_csv(OUT_DIR / "outlet_size_imputation_confusion_matrix.csv")

else:
    print("\nNot enough samples in one or more classes for cross-validation.")
    report_text = "Not enough samples for cross-validation."


# ============================================================
# FIT FINAL MODEL WITH CLASS BALANCING
# ============================================================

final_sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_enc
)

model.fit(
    X,
    y_enc,
    classifier__sample_weight=final_sample_weights
)


# ============================================================
# FEATURE IMPORTANCE
# ============================================================

try:
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
    importances = model.named_steps["classifier"].feature_importances_

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False)

    importance_df.to_csv(
        OUT_DIR / "outlet_size_imputation_feature_importance.csv",
        index=False
    )

    print("\nTop 20 feature importances:")
    print(importance_df.head(20))

except Exception as e:
    print("\nCould not extract feature importance:", e)


# ============================================================
# PREDICT MISSING OUTLET_SIZE
# ============================================================

merged["Outlet_Size_Imputed"] = 0
merged["Outlet_Size_Confidence"] = np.nan
merged["Outlet_Size_Review_Flag"] = "Original"

if len(missing) > 0:
    X_missing = missing[numeric_features + categorical_features]

    pred_enc = model.predict(X_missing)
    pred_labels = le.inverse_transform(pred_enc)

    pred_proba = model.predict_proba(X_missing)
    confidence = pred_proba.max(axis=1)

    merged.loc[missing.index, target_col] = pred_labels
    merged.loc[missing.index, "Outlet_Size_Imputed"] = 1
    merged.loc[missing.index, "Outlet_Size_Confidence"] = confidence

    merged.loc[missing.index, "Outlet_Size_Review_Flag"] = np.where(
        confidence < LOW_CONFIDENCE_THRESHOLD,
        "Low Confidence - Review",
        "Imputed OK"
    )

    low_confidence_rows = merged[
        (merged["Outlet_Size_Imputed"] == 1) &
        (merged["Outlet_Size_Confidence"] < LOW_CONFIDENCE_THRESHOLD)
    ].copy()

    low_confidence_rows.to_csv(
        OUT_DIR / "low_confidence_outlet_size_imputations.csv",
        index=False
    )

    print("\nMissing Outlet_Size values predicted.")
    print("Low-confidence imputations:", len(low_confidence_rows))

else:
    print("\nNo missing Outlet_Size values to predict.")


# ============================================================
# SAVE FINAL OUTPUTS
# ============================================================

# Full enriched output with imputation flags
merged.to_csv(
    OUT_DIR / "outlet_master_imputed_with_flags.csv",
    index=False
)

# Clean outlet master only with original columns
original_cols_available = [c for c in om.columns if c in merged.columns]

final_om = merged[original_cols_available].copy()

final_om.to_csv(
    OUT_DIR / "outlet_master_imputed.csv",
    index=False
)

# Diagnostics summary
with open(OUT_DIR / "outlet_size_imputation_diagnostics.txt", "w", encoding="utf-8") as f:
    f.write("Outlet Size Imputation Diagnostics\n")
    f.write("=================================\n\n")

    f.write(f"Target column: {target_col}\n")
    f.write(f"Join key: {key}\n")
    f.write(f"Training rows: {len(train)}\n")
    f.write(f"Missing rows predicted: {len(missing)}\n\n")

    f.write("Numeric features used:\n")
    for col in numeric_features:
        f.write(f"- {col}\n")

    f.write("\nCategorical features used:\n")
    for col in categorical_features:
        f.write(f"- {col}\n")

    f.write("\nClass distribution:\n")
    f.write(str(train[target_col].value_counts()))
    f.write("\n\nClass distribution %:\n")
    f.write(str((train[target_col].value_counts(normalize=True) * 100).round(2)))

    f.write("\n\nCross-validation report:\n")
    f.write(report_text)

print("\nSaved outputs:")
print(OUT_DIR / "outlet_master_imputed_with_flags.csv")
print(OUT_DIR / "outlet_master_imputed.csv")
print(OUT_DIR / "outlet_size_imputation_diagnostics.txt")
print(OUT_DIR / "outlet_size_imputation_cv_results.csv")
print(OUT_DIR / "outlet_size_imputation_confusion_matrix.csv")
print(OUT_DIR / "outlet_size_imputation_feature_importance.csv")
print(OUT_DIR / "low_confidence_outlet_size_imputations.csv")

Loaded files:
Outlet master: raw data\outlet_master.csv
Outlet coordinates: raw data\outlet_coordinates.csv
Transactions: raw data\transactions_history_final.csv

Target column: Outlet_Size
Join key: Outlet_ID

Transaction features added.

Numeric features used:
['Latitude', 'Longitude', 'Cooler_Count', 'avg_monthly_liters', 'median_monthly_liters', 'max_monthly_liters', 'p75_monthly_liters', 'p90_monthly_liters', 'sales_std', 'sales_cv', 'total_liters', 'avg_bill_value', 'max_bill_value', 'total_bill_value', 'avg_sku_count', 'max_sku_count', 'total_unique_skus', 'active_months', 'recent_3_month_avg_liters', 'recent_6_month_avg_liters', 'avg_january_liters', 'max_january_liters', 'distributor_count']

Categorical features used:
['Outlet_Type', 'Distributor_ID']

Training rows: 19804
Missing rows to predict: 196

Outlet_Size class distribution:
Outlet_Size
Small          10272
Medium          5702
Large           2887
Extra Large      943
Name: count, dtype: Int64

Outlet_Size class dis